# 🏠 PySpark MLlib — Regression Models on Boston Housing Dataset
  ## Industry-Level ML Pipeline: Linear Regression, Decision Tree & Gradient-Boosted Trees

  > **Fixes Applied:** Correct `!pip install`, fixed `JAVA_HOME`, modernised to `SparkSession`,  
  > replaced deprecated CSV reader, auto-downloads dataset, fixed truncated print, removed  
  > duplicate Decision Tree training, added visualisations & model comparison.

  ---
  ### 📌 Pipeline Overview
  | Step | Description |
  |------|-------------|
  | 1 | Environment setup & SparkSession |
  | 2 | Load & explore Boston Housing data |
  | 3 | Feature engineering with VectorAssembler |
  | 4 | Linear Regression — train, evaluate, visualise |
  | 5 | Decision Tree Regression — train & evaluate |
  | 6 | Gradient-Boosted Tree Regression — train & evaluate |
  | 7 | Model Comparison Dashboard |
  | 8 | Feature Importance Analysis |

---
## ⚙️ Step 1 — Install PySpark

In [ ]:
# ✅ FIX: Added '!' prefix so pip runs correctly in Colab/Jupyter
  !pip install pyspark -q
  print("✅ PySpark installed")

In [ ]:
import os
  # ✅ FIX: Corrected path from '/lib/jvm/' → '/usr/lib/jvm/'
  os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
  print(f"JAVA_HOME = {os.environ['JAVA_HOME']}")

In [ ]:
# ✅ FIX: Replaced deprecated SQLContext with modern SparkSession
  from pyspark.sql import SparkSession
  from pyspark.sql.functions import col, corr
  from pyspark.ml.feature import VectorAssembler
  from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor, GBTRegressor
  from pyspark.ml.evaluation import RegressionEvaluator
  import matplotlib.pyplot as plt
  import matplotlib.patches as mpatches
  import pandas as pd
  import numpy as np

  print("✅ All imports successful")

---
## 🚀 Step 2 — Initialize SparkSession

In [ ]:
# ✅ FIX: SparkSession is the modern unified entry point (SparkContext + SQLContext + HiveContext)
  spark = SparkSession.builder \
      .appName("BostonHousing_Regression_MLlib") \
      .master("local[*]") \
      .config("spark.ui.port", "4050") \
      .config("spark.executor.memory", "2g") \
      .config("spark.driver.memory", "2g") \
      .config("spark.sql.shuffle.partitions", "8") \
      .getOrCreate()

  spark.sparkContext.setLogLevel("ERROR")
  print(f"✅ SparkSession created")
  print(f"   App Name : {spark.sparkContext.appName}")
  print(f"   Version  : {spark.version}")
  print(f"   Master   : {spark.sparkContext.master}")

---
## 📂 Step 3 — Load Boston Housing Dataset

> ✅ **FIX:** Auto-downloads the dataset from a public URL — no manual CSV upload needed.

In [ ]:
import urllib.request

  # ✅ FIX: Auto-download dataset — original notebook had no download step
  DATASET_URL = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
  LOCAL_PATH  = "/content/Boston.csv"

  try:
      urllib.request.urlretrieve(DATASET_URL, LOCAL_PATH)
      print(f"✅ Dataset downloaded → {LOCAL_PATH}")
  except Exception as e:
      # Fallback: generate a realistic synthetic dataset if URL fails
      print(f"⚠️  URL failed ({e}). Generating synthetic Boston Housing data...")
      import numpy as np, pandas as pd
      np.random.seed(42)
      n = 506
      synthetic = pd.DataFrame({
          'crim':    np.abs(np.random.exponential(3.6, n)),
          'zn':      np.random.choice([0,12.5,25,50,75,100], n),
          'indus':   np.random.uniform(0.46, 27.74, n),
          'chas':    np.random.choice([0, 1], n, p=[0.93, 0.07]),
          'nox':     np.random.uniform(0.385, 0.871, n),
          'rm':      np.random.normal(6.28, 0.70, n).clip(3.56, 8.78),
          'age':     np.random.uniform(2.9, 100.0, n),
          'dis':     np.random.uniform(1.13, 12.13, n),
          'rad':     np.random.choice([1,2,3,4,5,6,7,8,24], n),
          'tax':     np.random.choice([193,222,242,254,270,273,277,307,330,384,432,666,711], n),
          'ptratio': np.random.uniform(12.6, 22.0, n),
          'black':   np.random.uniform(0.32, 396.9, n),
          'lstat':   np.random.uniform(1.73, 37.97, n),
      })
      synthetic['medv'] = (
          -0.1 * synthetic['crim'] + 0.05 * synthetic['zn'] -
          0.04 * synthetic['indus'] + 2.7 * synthetic['chas'] -
          17.8 * synthetic['nox'] + 3.8 * synthetic['rm'] -
          0.006 * synthetic['age'] - 1.5 * synthetic['dis'] +
          0.3 * synthetic['rad'] - 0.012 * synthetic['tax'] -
          0.95 * synthetic['ptratio'] + 0.009 * synthetic['black'] -
          0.52 * synthetic['lstat'] + 36.5 +
          np.random.normal(0, 1.5, n)
      ).clip(5, 50)
      synthetic.to_csv(LOCAL_PATH, index=False)
      print(f"✅ Synthetic dataset saved → {LOCAL_PATH}")

In [ ]:
# ✅ FIX: Replaced deprecated 'com.databricks.spark.csv' with native spark.read.csv()
  house_df = spark.read.csv(LOCAL_PATH, header=True, inferSchema=True)

  print(f"✅ Dataset loaded: {house_df.count()} rows × {len(house_df.columns)} columns")
  house_df.show(5)

---
## 🔍 Step 4 — Exploratory Data Analysis

In [ ]:
print("📋 Schema:")
  house_df.printSchema()

In [ ]:
print("📊 Descriptive Statistics:")
  house_df.describe().toPandas().set_index('summary').T

In [ ]:
# 🆕 ADDED: EDA visualisations — not in original notebook
  pdf = house_df.toPandas()

  fig, axes = plt.subplots(2, 3, figsize=(16, 9))
  fig.patch.set_facecolor('#0f0f1a')
  fig.suptitle('Boston Housing — Feature Distributions & Correlations', fontsize=14,
               fontweight='bold', color='white')

  features_to_plot = ['rm', 'lstat', 'crim', 'nox', 'dis', 'ptratio']
  palette = ['#2196F3','#4CAF50','#FF5722','#FF9800','#9C27B0','#00BCD4']

  for ax, feat, color in zip(axes.flatten(), features_to_plot, palette):
      ax.set_facecolor('#1a1a2e')
      ax.scatter(pdf[feat], pdf['medv'], alpha=0.4, s=12, color=color)
      z = np.polyfit(pdf[feat], pdf['medv'], 1)
      p = np.poly1d(z)
      xs = np.linspace(pdf[feat].min(), pdf[feat].max(), 100)
      ax.plot(xs, p(xs), color='white', lw=1.5, linestyle='--')
      corr_val = pdf[feat].corr(pdf['medv'])
      ax.set_title(f'{feat} vs medv  (r={corr_val:.2f})', color='white', fontsize=9)
      ax.set_xlabel(feat, color='#aaa', fontsize=8)
      ax.set_ylabel('medv ($000)', color='#aaa', fontsize=8)
      ax.tick_params(colors='white', labelsize=7)
      for spine in ax.spines.values(): spine.set_edgecolor('#444')

  plt.tight_layout()
  plt.savefig('eda_scatter.png', dpi=120, bbox_inches='tight', facecolor=fig.get_facecolor())
  plt.show()
  print("✅ EDA scatter plots saved")

---
## 🔧 Step 5 — Feature Engineering with VectorAssembler

In [ ]:
FEATURE_COLS = ['crim','zn','indus','chas','nox','rm','age','dis','rad','tax','ptratio','black','lstat']
  LABEL_COL    = 'medv'

  assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol='features')
  vhouse_df = assembler.transform(house_df).select(['features', LABEL_COL])

  print("✅ VectorAssembler applied")
  print(f"   Feature vector length : {len(FEATURE_COLS)}")
  vhouse_df.show(3)

In [ ]:
# Reproducible train/test split
  train_df, test_df = vhouse_df.randomSplit([0.7, 0.3], seed=42)
  print(f"✅ Train/Test split (70/30, seed=42)")
  print(f"   Train records : {train_df.count()}")
  print(f"   Test  records : {test_df.count()}")

---
## 📈 Step 6 — Linear Regression

A parametric baseline model that fits a hyperplane through the feature space.

In [ ]:
lr = LinearRegression(
      featuresCol='features', labelCol=LABEL_COL,
      maxIter=100,          # increased from 10 for better convergence
      regParam=0.01,        # L2 regularisation
      elasticNetParam=0.0,  # pure Ridge
  )
  lr_model = lr.fit(train_df)

  print("✅ Linear Regression model trained")
  print(f"\n   Coefficients  : {lr_model.coefficients}")
  print(f"   Intercept     : {lr_model.intercept:.4f}")

In [ ]:
# Training metrics
  summary = lr_model.summary
  print("📊 Training Summary")
  print(f"   Train RMSE : {summary.rootMeanSquaredError:.4f}")
  print(f"   Train R²   : {summary.r2:.4f}")
  print(f"   Iterations : {summary.totalIterations}")
  print(f"   Residuals (first 5 rows):")
  summary.residuals.show(5)

  # Test metrics
  lr_preds = lr_model.transform(test_df)
  lr_eval_r2   = RegressionEvaluator(predictionCol="prediction", labelCol=LABEL_COL, metricName="r2")
  lr_eval_rmse = RegressionEvaluator(predictionCol="prediction", labelCol=LABEL_COL, metricName="rmse")
  lr_eval_mae  = RegressionEvaluator(predictionCol="prediction", labelCol=LABEL_COL, metricName="mae")

  lr_r2   = lr_eval_r2.evaluate(lr_preds)
  lr_rmse = lr_eval_rmse.evaluate(lr_preds)
  lr_mae  = lr_eval_mae.evaluate(lr_preds)

  print(f"\n📊 Test Evaluation")
  print(f"   Test R²   : {lr_r2:.4f}")
  print(f"   Test RMSE : {lr_rmse:.4f}")
  print(f"   Test MAE  : {lr_mae:.4f}")

  lr_preds.select("prediction", LABEL_COL).show(5)

In [ ]:
# 🆕 ADDED: Prediction vs Actual + Residuals plot for Linear Regression
  lr_pdf = lr_preds.toPandas()
  residuals = lr_pdf['medv'] - lr_pdf['prediction']

  fig, axes = plt.subplots(1, 3, figsize=(17, 5))
  fig.patch.set_facecolor('#0f0f1a')
  fig.suptitle('Linear Regression — Diagnostics', fontsize=13, fontweight='bold', color='white')

  # Chart 1: Predicted vs Actual
  ax1 = axes[0]; ax1.set_facecolor('#1a1a2e')
  ax1.scatter(lr_pdf['medv'], lr_pdf['prediction'], alpha=0.5, s=18, color='#2196F3')
  lims = [min(lr_pdf['medv'].min(), lr_pdf['prediction'].min()),
          max(lr_pdf['medv'].max(), lr_pdf['prediction'].max())]
  ax1.plot(lims, lims, 'w--', lw=1.5, label='Perfect fit')
  ax1.set_xlabel('Actual medv', color='white'); ax1.set_ylabel('Predicted medv', color='white')
  ax1.set_title(f'Predicted vs Actual\nR² = {lr_r2:.3f}', color='white')
  ax1.tick_params(colors='white'); ax1.legend(labelcolor='white', facecolor='#0f0f1a')
  for s in ax1.spines.values(): s.set_edgecolor('#444')

  # Chart 2: Residuals vs Predicted
  ax2 = axes[1]; ax2.set_facecolor('#1a1a2e')
  ax2.scatter(lr_pdf['prediction'], residuals, alpha=0.5, s=18, color='#FF9800')
  ax2.axhline(0, color='white', lw=1.5, linestyle='--')
  ax2.set_xlabel('Predicted medv', color='white'); ax2.set_ylabel('Residuals', color='white')
  ax2.set_title('Residuals vs Predicted', color='white')
  ax2.tick_params(colors='white')
  for s in ax2.spines.values(): s.set_edgecolor('#444')

  # Chart 3: Residuals histogram
  ax3 = axes[2]; ax3.set_facecolor('#1a1a2e')
  ax3.hist(residuals, bins=25, color='#4CAF50', edgecolor='#1a1a2e', alpha=0.85)
  ax3.axvline(0, color='white', lw=1.5, linestyle='--')
  ax3.set_xlabel('Residual', color='white'); ax3.set_ylabel('Count', color='white')
  ax3.set_title('Residuals Distribution', color='white')
  ax3.tick_params(colors='white')
  for s in ax3.spines.values(): s.set_edgecolor('#444')

  plt.tight_layout()
  plt.savefig('lr_diagnostics.png', dpi=120, bbox_inches='tight', facecolor=fig.get_facecolor())
  plt.show()

---
## 🌳 Step 7 — Decision Tree Regression

A non-parametric tree-based model that recursively partitions the feature space into regions.

In [ ]:
dt = DecisionTreeRegressor(
      featuresCol='features', labelCol=LABEL_COL,
      maxDepth=5,           # controls overfitting
      minInstancesPerNode=5,
      seed=42,
  )
  dt_model = dt.fit(train_df)
  dt_preds  = dt_model.transform(test_df)

  dt_r2   = RegressionEvaluator(predictionCol="prediction", labelCol=LABEL_COL, metricName="r2").evaluate(dt_preds)
  dt_rmse = RegressionEvaluator(predictionCol="prediction", labelCol=LABEL_COL, metricName="rmse").evaluate(dt_preds)
  dt_mae  = RegressionEvaluator(predictionCol="prediction", labelCol=LABEL_COL, metricName="mae").evaluate(dt_preds)

  # ✅ FIX: Removed duplicate Decision Tree training block (was trained twice in original)
  # ✅ FIX: Added all three metrics in one block instead of scattered cells
  print("✅ Decision Tree model trained")
  print(f"\n📊 Test Evaluation")
  print(f"   Test R²   : {dt_r2:.4f}")
  print(f"   Test RMSE : {dt_rmse:.4f}")  # ✅ FIX: was truncated in original
  print(f"   Test MAE  : {dt_mae:.4f}")

  dt_preds.select("prediction", LABEL_COL).show(5)

In [ ]:
# ✅ FIX: Original had 'dt_model.featureImportances' with no print — silent output
  # 🆕 ADDED: Proper feature importance bar chart
  importances = dt_model.featureImportances.toArray()
  fi_df = pd.DataFrame({'feature': FEATURE_COLS, 'importance': importances})
  fi_df = fi_df.sort_values('importance', ascending=True)

  print("📊 Decision Tree Feature Importances:")
  for _, row in fi_df.sort_values('importance', ascending=False).iterrows():
      bar = '█' * int(row['importance'] * 80)
      print(f"   {row['feature']:<10}: {row['importance']:.4f}  {bar}")

  fig, ax = plt.subplots(figsize=(10, 6))
  ax.set_facecolor('#1a1a2e'); fig.patch.set_facecolor('#0f0f1a')
  colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(fi_df)))
  bars = ax.barh(fi_df['feature'], fi_df['importance'], color=colors, edgecolor='#333')
  for bar, val in zip(bars, fi_df['importance']):
      ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
              f'{val:.3f}', va='center', fontsize=8, color='white')
  ax.set_xlabel('Importance Score', color='white')
  ax.set_title('Decision Tree — Feature Importance', color='white', fontsize=12, fontweight='bold')
  ax.tick_params(colors='white')
  for s in ax.spines.values(): s.set_edgecolor('#444')
  plt.tight_layout()
  plt.savefig('dt_feature_importance.png', dpi=120, bbox_inches='tight', facecolor=fig.get_facecolor())
  plt.show()

In [ ]:
# 🆕 ADDED: Prediction vs Actual scatter for Decision Tree
  dt_pdf = dt_preds.toPandas()

  fig, ax = plt.subplots(figsize=(7, 6))
  ax.set_facecolor('#1a1a2e'); fig.patch.set_facecolor('#0f0f1a')
  ax.scatter(dt_pdf['medv'], dt_pdf['prediction'], alpha=0.5, s=20, color='#FF9800')
  lims = [min(dt_pdf['medv'].min(), dt_pdf['prediction'].min()),
          max(dt_pdf['medv'].max(), dt_pdf['prediction'].max())]
  ax.plot(lims, lims, 'w--', lw=1.5)
  ax.set_xlabel('Actual medv', color='white'); ax.set_ylabel('Predicted medv', color='white')
  ax.set_title(f'Decision Tree — Predicted vs Actual\nR² = {dt_r2:.3f}  RMSE = {dt_rmse:.3f}',
               color='white', fontsize=11)
  ax.tick_params(colors='white')
  for s in ax.spines.values(): s.set_edgecolor('#444')
  plt.tight_layout()
  plt.show()

---
## 🚀 Step 8 — Gradient-Boosted Tree (GBT) Regression

An ensemble method that sequentially adds weak learners, each correcting the errors of the previous. Generally the strongest of the three models.

In [ ]:
gbt = GBTRegressor(
      featuresCol='features', labelCol=LABEL_COL,
      maxIter=100,           # number of boosting rounds
      maxDepth=5,
      stepSize=0.1,          # learning rate
      subsamplingRate=0.8,   # row subsampling (like sklearn's subsample)
      seed=42,
  )
  gbt_model = gbt.fit(train_df)
  gbt_preds  = gbt_model.transform(test_df)

  gbt_r2   = RegressionEvaluator(predictionCol="prediction", labelCol=LABEL_COL, metricName="r2").evaluate(gbt_preds)
  gbt_rmse = RegressionEvaluator(predictionCol="prediction", labelCol=LABEL_COL, metricName="rmse").evaluate(gbt_preds)
  gbt_mae  = RegressionEvaluator(predictionCol="prediction", labelCol=LABEL_COL, metricName="mae").evaluate(gbt_preds)

  print("✅ GBT model trained")
  print(f"\n📊 Test Evaluation")
  print(f"   Test R²   : {gbt_r2:.4f}")
  print(f"   Test RMSE : {gbt_rmse:.4f}")
  print(f"   Test MAE  : {gbt_mae:.4f}")

  gbt_preds.select('prediction', LABEL_COL).show(5)

In [ ]:
# 🆕 ADDED: GBT Feature Importance
  gbt_importances = gbt_model.featureImportances.toArray()
  gbt_fi = pd.DataFrame({'feature': FEATURE_COLS, 'importance': gbt_importances})
  gbt_fi = gbt_fi.sort_values('importance', ascending=True)

  fig, axes = plt.subplots(1, 2, figsize=(16, 6))
  fig.patch.set_facecolor('#0f0f1a')
  fig.suptitle('GBT — Feature Importance & Predicted vs Actual', fontsize=13, fontweight='bold', color='white')

  # Feature importance
  ax1 = axes[0]; ax1.set_facecolor('#1a1a2e')
  cols = plt.cm.viridis(np.linspace(0.2, 0.9, len(gbt_fi)))
  bars = ax1.barh(gbt_fi['feature'], gbt_fi['importance'], color=cols, edgecolor='#333')
  for bar, val in zip(bars, gbt_fi['importance']):
      ax1.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
               f'{val:.3f}', va='center', fontsize=8, color='white')
  ax1.set_xlabel('Importance', color='white')
  ax1.set_title('Feature Importance', color='white')
  ax1.tick_params(colors='white')
  for s in ax1.spines.values(): s.set_edgecolor('#444')

  # Predicted vs Actual
  gbt_pdf = gbt_preds.toPandas()
  ax2 = axes[1]; ax2.set_facecolor('#1a1a2e')
  ax2.scatter(gbt_pdf['medv'], gbt_pdf['prediction'], alpha=0.5, s=18, color='#E91E63')
  lims = [min(gbt_pdf['medv'].min(), gbt_pdf['prediction'].min()),
          max(gbt_pdf['medv'].max(), gbt_pdf['prediction'].max())]
  ax2.plot(lims, lims, 'w--', lw=1.5)
  ax2.set_xlabel('Actual medv', color='white'); ax2.set_ylabel('Predicted medv', color='white')
  ax2.set_title(f'Predicted vs Actual\nR² = {gbt_r2:.3f}  RMSE = {gbt_rmse:.3f}', color='white')
  ax2.tick_params(colors='white')
  for s in ax2.spines.values(): s.set_edgecolor('#444')

  plt.tight_layout()
  plt.savefig('gbt_analysis.png', dpi=120, bbox_inches='tight', facecolor=fig.get_facecolor())
  plt.show()

---
## 🏆 Step 9 — Model Comparison Dashboard

In [ ]:
# 🆕 ADDED: Comprehensive model comparison — not in original notebook at all
  models     = ['Linear Regression', 'Decision Tree', 'GBT']
  r2_scores  = [lr_r2,   dt_r2,   gbt_r2]
  rmse_scores= [lr_rmse, dt_rmse, gbt_rmse]
  mae_scores = [lr_mae,  dt_mae,  gbt_mae]
  colors_bar = ['#2196F3', '#FF9800', '#E91E63']

  fig, axes = plt.subplots(1, 3, figsize=(17, 6))
  fig.patch.set_facecolor('#0f0f1a')
  fig.suptitle('🏆 Model Comparison Dashboard — Boston Housing Regression',
               fontsize=14, fontweight='bold', color='white')

  metrics = [
      (axes[0], 'R² Score (higher = better)',  r2_scores,   True),
      (axes[1], 'RMSE      (lower  = better)', rmse_scores, False),
      (axes[2], 'MAE       (lower  = better)', mae_scores,  False),
  ]
  for ax, title, vals, higher_better in metrics:
      ax.set_facecolor('#1a1a2e')
      bars = ax.bar(models, vals, color=colors_bar, edgecolor='#333', width=0.5)
      best_idx = vals.index(max(vals) if higher_better else min(vals))
      bars[best_idx].set_edgecolor('gold'); bars[best_idx].set_linewidth(3)
      for bar, v in zip(bars, vals):
          ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                  f'{v:.3f}', ha='center', va='bottom', fontsize=10, color='white', fontweight='bold')
      ax.set_title(title, color='white', fontsize=10)
      ax.tick_params(colors='white', axis='y')
      ax.set_xticks(range(len(models)))
      ax.set_xticklabels(models, color='white', fontsize=8, rotation=10)
      for s in ax.spines.values(): s.set_edgecolor('#444')
      ax.text(bars[best_idx].get_x() + bars[best_idx].get_width()/2,
              bars[best_idx].get_height() * 0.5, '🏆', ha='center', va='center', fontsize=16)

  plt.tight_layout()
  plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight', facecolor=fig.get_facecolor())
  plt.show()

In [ ]:
# 🆕 ADDED: Printed summary table
  print("=" * 60)
  print("   FINAL MODEL COMPARISON SUMMARY")
  print("=" * 60)
  print(f"{'Model':<22} {'R²':>8} {'RMSE':>8} {'MAE':>8}")
  print("-" * 60)
  for m, r2, rmse, mae in zip(models, r2_scores, rmse_scores, mae_scores):
      best_r2   = "⭐" if r2   == max(r2_scores)   else "  "
      best_rmse = "⭐" if rmse == min(rmse_scores) else "  "
      best_mae  = "⭐" if mae  == min(mae_scores)  else "  "
      print(f"{m:<22} {r2:>7.4f}{best_r2} {rmse:>7.4f}{best_rmse} {mae:>7.4f}{best_mae}")
  print("=" * 60)
  print("\n⭐ = best score in that column")
  print(f"\n✅ Best model overall: {models[r2_scores.index(max(r2_scores))]}")

---
## 💾 Step 10 — Save Best Model Predictions

In [ ]:
# Save GBT predictions to CSV (best model)
  gbt_preds.select('prediction', LABEL_COL) \
      .withColumnRenamed('prediction', 'predicted_medv') \
      .toPandas() \
      .to_csv('/content/gbt_predictions.csv', index=False)
  print("✅ GBT predictions saved → /content/gbt_predictions.csv")

  # Clean shutdown
  spark.stop()
  print("✅ SparkSession stopped cleanly")

---
  ## 📖 MLlib Regression API Cheat Sheet

  | Class | Key Parameters | Notes |
  |-------|---------------|-------|
  | `LinearRegression` | `maxIter`, `regParam`, `elasticNetParam` | Ridge (0), Lasso (1), ElasticNet (0-1) |
  | `DecisionTreeRegressor` | `maxDepth`, `minInstancesPerNode` | Prone to overfitting at high depth |
  | `GBTRegressor` | `maxIter`, `maxDepth`, `stepSize` | Usually best; slower to train |
  | `RegressionEvaluator` | `metricName` = r2/rmse/mae/mse | Always evaluate on **held-out** test set |
  | `VectorAssembler` | `inputCols`, `outputCol` | Required to pack features into a vector |

  ### 🛠️ All Issues Fixed in This Notebook
  | # | Original Bug | Fix Applied |
  |---|-------------|-------------|
  | 1 | `pip install` missing `!` | Added `!` prefix |
  | 2 | Wrong `JAVA_HOME` path | `/lib/jvm/` → `/usr/lib/jvm/` |
  | 3 | Deprecated `SQLContext` | Replaced with `SparkSession` |
  | 4 | Deprecated `com.databricks.spark.csv` | Native `spark.read.csv()` |
  | 5 | No dataset download step | Auto-downloads from public URL |
  | 6 | Truncated print statement (Cell 15) | Fixed with full string |
  | 7 | Decision Tree trained twice | Removed duplicate training block |
  | 8 | Silent `featureImportances` output | Added `print()` + bar chart |
  | 9 | Empty last cell | Replaced with clean shutdown |
  | 10 | No visualisations | 6 charts added |